# Lista 09 · Parsers de texto

O log de verdade chega em formatos diferentes, porque a rede tem fabricantes
diferentes. Esta lista constrói, peça por peça, o parser do capítulo 9: primeiro as
ferramentas (`partition`, `split` com `maxsplit`, tabelas de tradução), depois um
leitor para cada formato, e por fim a validação de endereços e o resumo de um log
misturado.

Os três formatos, para ter à mão:

```
A  2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3
B  Mar  2 14:05:10 SWITCH-NORTE-02 %LINK-3-UPDOWN: Interface Gi0/12, changed state to down
C  ts=2026-03-02T14:07:44 host=RADIO-OESTE-01 sev=warn msg="enlace degradado"
```

Todo leitor devolve o **mesmo dicionário**, com as chaves `momento`
(`"AAAA-MM-DD HH:MM:SS"`), `severidade`, `equipamento` e `mensagem`.

O exercício 10 traz os três leitores já escritos: o trabalho é juntá-los.

---

**Como usar este caderno:** cada exercício tem duas células. Na primeira,
escreva a sua solução no lugar do `# TODO`. A segunda tem os testes —
rode-a e ela diz se a sua função está correta. Não altere a célula de teste.

Se um teste falhar, o Python mostra um `AssertionError` apontando a linha:
é aquele caso específico que a sua função ainda não atende.

Termo de telecom estranho no enunciado? Veja
[A rede da Maré Net](https://lacouth.github.io/python_telecom-site/unidade0-primeiros-passos/rede-marenet/) ou o
[glossário](https://lacouth.github.io/python_telecom-site/apendices/glossario/).

**Comece pela célula abaixo.** Ela cria os arquivos de exemplo que os
exercícios desta lista leem. Sem ela, os testes falham com
`FileNotFoundError`.

In [ ]:
"""Cria o log misturado que o exercício 10 lê.

Rode esta célula antes de tudo. Ela não baixa nada: escreve o arquivo direto na
pasta de trabalho da sessão.
"""

LOG = """2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3
Mar  2 14:05:10 SWITCH-NORTE-02 %LINK-3-UPDOWN: Interface Gi0/12, changed state to down
ts=2026-03-02T14:07:44 host=RADIO-OESTE-01 sev=warn msg="enlace degradado"
-- MARK --
Mar  2 14:20:31 SWITCH-NORTE-02 %LINK-X-UPDOWN: Interface Gi0/12, changed state to up
2026-03-02 14:31:02 WARNING OLT-SUL-03 potencia optica degradada: -27.2 dBm
ts=2026-03-02T14:40:00 host=RADIO-OESTE-01
Mar  2 14:52:16 SWITCH-CENTRO-01 %SYS-2-MALLOCFAIL: Memory allocation of 4096 bytes failed
2026-03-02 15:0
ts=2026-03-02T15:10:09 host=RADIO-OESTE-01 sev=crit msg="enlace fora do ar"

"""

with open("alarmes_misturados.log", "w", encoding="utf-8") as arquivo:
    arquivo.write(LOG)

print("alarmes_misturados.log criado")

### Exercício 01

Linhas no formato chave=valor trazem pares como `host=RADIO-OESTE-01`. Escreva
`valor_do_par(par, chave)`, que devolve o valor se o par for daquela chave, ou
`None` se for de outra chave ou não tiver `=`. Use `partition`.

```python
valor_do_par("host=RADIO-OESTE-01", "host")   # -> "RADIO-OESTE-01"
valor_do_par("sev=warn", "host")              # -> None
valor_do_par("-- MARK --", "host")            # -> None
```

In [ ]:
def valor_do_par(par, chave):
    """Valor do par chave=valor, ou None se a chave for outra."""
    # TODO: nome, separador, valor = par.partition("=")
    #       compare nome com a chave (e confira que o separador veio)
    pass

In [ ]:
# Célula de teste — Exercício 01
assert valor_do_par("host=RADIO-OESTE-01", "host") == "RADIO-OESTE-01"
assert valor_do_par("sev=warn", "sev") == "warn"
assert valor_do_par("sev=warn", "host") is None
assert valor_do_par("-- MARK --", "host") is None
assert valor_do_par("url=http://x/?a=1", "url") == "http://x/?a=1"
assert valor_do_par("host=", "host") == ""
print("Exercício 01: todos os testes passaram!")

### Exercício 02

Escreva `mensagem_do_formato_a(linha)`, que devolve a mensagem de uma linha no
formato A — tudo o que vem depois do nome do equipamento, **com os espaços
originais**. Use `split` com `maxsplit`.

```python
mensagem_do_formato_a("2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal")
# -> "perda de sinal"
```

In [ ]:
def mensagem_do_formato_a(linha):
    """A mensagem de uma linha do formato A (do quinto campo em diante)."""
    # TODO: linha.split(maxsplit=4) deixa a mensagem inteira na posição 4
    pass

In [ ]:
# Célula de teste — Exercício 02
assert mensagem_do_formato_a("2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3") == "perda de sinal na porta GPON0/1/3"
assert mensagem_do_formato_a("2026-03-02 09:00:00 INFO SWITCH-NORTE-02 porta ativada") == "porta ativada"
# dois espaços dentro da mensagem continuam lá
assert mensagem_do_formato_a("2026-03-02 09:00:00 INFO X valor:  -21.4") == "valor:  -21.4"
print("Exercício 02: todos os testes passaram!")

### Exercício 03

No formato B, a severidade vem como um número dentro do código, como em
`%LINK-3-UPDOWN`. Escreva `severidade_do_codigo(codigo)`, que devolve o **nome** da
severidade usando a tabela `SEVERIDADE_POR_NUMERO` (já no esqueleto), ou `None` se
o número não estiver na tabela.

```python
severidade_do_codigo("%LINK-3-UPDOWN")     # -> "ERROR"
severidade_do_codigo("%SYS-5-CONFIG_I")    # -> "NOTICE"
severidade_do_codigo("%LINK-X-UPDOWN")     # -> None
```

In [ ]:
SEVERIDADE_POR_NUMERO = {"0": "EMERGENCY", "1": "ALERT", "2": "CRITICAL", "3": "ERROR",
                         "4": "WARNING", "5": "NOTICE", "6": "INFO", "7": "DEBUG"}


def severidade_do_codigo(codigo):
    """Nome da severidade de um código %FACILIDADE-N-MNEMONICO, ou None."""
    # TODO: tire o "%" com strip, quebre com split("-") e consulte a tabela
    #       com .get(), para devolver None quando o número não existir
    pass

In [ ]:
# Célula de teste — Exercício 03
assert severidade_do_codigo("%LINK-3-UPDOWN") == "ERROR"
assert severidade_do_codigo("%SYS-5-CONFIG_I") == "NOTICE"
assert severidade_do_codigo("%SYS-2-MALLOCFAIL") == "CRITICAL"
assert severidade_do_codigo("%ENVMON-4-TEMP") == "WARNING"
assert severidade_do_codigo("%LINK-X-UPDOWN") is None
assert severidade_do_codigo("%LINK-9-UPDOWN") is None
print("Exercício 03: todos os testes passaram!")

### Exercício 04

Escreva `detecta_formato(linha)`, que devolve `"A"`, `"B"`, `"C"` ou `None`:

- **C** começa com `ts=`;
- **B** começa com um mês abreviado em inglês (uma chave de `MESES`, já no
  esqueleto) **e** contém `%`;
- **A** começa com quatro algarismos seguidos de hífen.

In [ ]:
MESES = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
         "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}


def detecta_formato(linha):
    """Diz de qual fabricante é a linha: "A", "B", "C" ou None."""
    # TODO: um if para cada formato, do sinal mais específico ao mais genérico.
    #       startswith, linha[:3] in MESES, linha[:4].isdigit() e linha[4:5]
    pass

In [ ]:
# Célula de teste — Exercício 04
assert detecta_formato("2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3") == "A"
assert detecta_formato("Mar  2 14:05:10 SWITCH-NORTE-02 %LINK-3-UPDOWN: Interface Gi0/12, changed state to down") == "B"
assert detecta_formato('ts=2026-03-02T14:07:44 host=RADIO-OESTE-01 sev=warn msg="enlace degradado"') == "C"
assert detecta_formato("-- MARK --") is None
assert detecta_formato("") is None
assert detecta_formato("Marcos esteve aqui") is None       # começa com "Mar", mas sem %
assert detecta_formato("2026 foi um bom ano") is None       # quatro algarismos, sem hífen
print("Exercício 04: todos os testes passaram!")

### Exercício 05

Escreva `le_formato_a(linha)`, que devolve o dicionário normalizado de uma linha
do formato A, com as chaves `momento`, `severidade`, `equipamento` e `mensagem`.

```python
le_formato_a("2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal")
# -> {"momento": "2026-03-02 14:03:17", "severidade": "CRITICAL",
#     "equipamento": "OLT-CENTRO-01", "mensagem": "perda de sinal"}
```

In [ ]:
def le_formato_a(linha):
    """Formato A: 2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 mensagem..."""
    # TODO: split(maxsplit=4) e monte o dicionário; o momento é data + " " + hora
    pass

In [ ]:
# Célula de teste — Exercício 05
esperado = {"momento": "2026-03-02 14:03:17", "severidade": "CRITICAL",
            "equipamento": "OLT-CENTRO-01", "mensagem": "perda de sinal na porta GPON0/1/3"}
assert le_formato_a("2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3") == esperado
alarme = le_formato_a("2026-03-02 09:00:00 INFO SWITCH-NORTE-02 porta ativada")
assert alarme["severidade"] == "INFO"
assert alarme["mensagem"] == "porta ativada"
print("Exercício 05: todos os testes passaram!")

### Exercício 06

Escreva `le_formato_b(linha, ano=2026)` para o formato B (estilo syslog Cisco).
As tabelas já estão no esqueleto. Cuidado com três coisas: o ano não vem na linha;
o mês vem abreviado; e há **dois espaços** antes do dia de um algarismo.

```python
le_formato_b("Mar  2 14:05:10 SWITCH-NORTE-02 %LINK-3-UPDOWN: Interface Gi0/12, changed state to down")
# -> {"momento": "2026-03-02 14:05:10", "severidade": "ERROR",
#     "equipamento": "SWITCH-NORTE-02",
#     "mensagem": "Interface Gi0/12, changed state to down"}
```

In [ ]:
MESES = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
         "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
SEVERIDADE_POR_NUMERO = {"0": "EMERGENCY", "1": "ALERT", "2": "CRITICAL", "3": "ERROR",
                         "4": "WARNING", "5": "NOTICE", "6": "INFO", "7": "DEBUG"}
SEVERIDADE_C = {"crit": "CRITICAL", "warn": "WARNING", "info": "INFO"}


def le_formato_b(linha, ano=2026):
    """Formato B: Mar  2 14:05:10 SWITCH-NORTE-02 %LINK-3-UPDOWN: texto"""
    # TODO:
    #   1. campos = linha.split(maxsplit=4)   (split() ignora os espaços duplos)
    #   2. mês pela tabela MESES; dia com int()
    #   3. código e mensagem: campos[4].partition(": ")
    #   4. severidade: o número do meio do código, pela tabela
    #   5. momento: f"{ano}-{mes:02d}-{dia:02d} {hora}"
    pass

In [ ]:
# Célula de teste — Exercício 06
esperado = {"momento": "2026-03-02 14:05:10", "severidade": "ERROR",
            "equipamento": "SWITCH-NORTE-02",
            "mensagem": "Interface Gi0/12, changed state to down"}
assert le_formato_b("Mar  2 14:05:10 SWITCH-NORTE-02 %LINK-3-UPDOWN: Interface Gi0/12, changed state to down") == esperado
alarme = le_formato_b("Dec 31 23:59:58 SWITCH-CENTRO-01 %SYS-5-CONFIG_I: Configured from console by admin", ano=2025)
assert alarme["momento"] == "2025-12-31 23:59:58"
assert alarme["severidade"] == "NOTICE"
assert alarme["mensagem"] == "Configured from console by admin"
# a mensagem pode ter ": " dentro -- só o primeiro separa o código
alarme = le_formato_b("Mar  2 03:15:16 SWITCH-NORTE-02 %ENVMON-4-TEMP: Sensor: 66 C")
assert alarme["mensagem"] == "Sensor: 66 C"
print("Exercício 06: todos os testes passaram!")

### Exercício 07

Escreva `le_formato_c(linha)` para o formato chave=valor. A mensagem vem entre
aspas e **tem espaços dentro** — separe-a antes de quebrar o resto. A tabela
`SEVERIDADE_C` já está no esqueleto.

```python
le_formato_c('ts=2026-03-02T14:07:44 host=RADIO-OESTE-01 sev=warn msg="enlace degradado"')
# -> {"momento": "2026-03-02 14:07:44", "severidade": "WARNING",
#     "equipamento": "RADIO-OESTE-01", "mensagem": "enlace degradado"}
```

In [ ]:
SEVERIDADE_C = {"crit": "CRITICAL", "warn": "WARNING", "info": "INFO"}


def le_formato_c(linha):
    """Formato C: ts=... host=... sev=... msg="texto" """
    # TODO:
    #   1. antes, _, mensagem = linha.partition(" msg=")
    #   2. um dicionário campos: para cada par em antes.split(), partition("=")
    #   3. o momento troca o "T" por espaço; a mensagem perde as aspas (strip)
    pass

In [ ]:
# Célula de teste — Exercício 07
esperado = {"momento": "2026-03-02 14:07:44", "severidade": "WARNING",
            "equipamento": "RADIO-OESTE-01", "mensagem": "enlace degradado"}
assert le_formato_c('ts=2026-03-02T14:07:44 host=RADIO-OESTE-01 sev=warn msg="enlace degradado"') == esperado
alarme = le_formato_c('ts=2026-03-02T15:10:09 host=RADIO-OESTE-01 sev=crit msg="enlace fora do ar"')
assert alarme["severidade"] == "CRITICAL"
assert alarme["mensagem"] == "enlace fora do ar"
# a ordem dos pares não importa
alarme = le_formato_c('host=RADIO-02 ts=2026-03-02T01:00:00 sev=info msg="modulacao 64QAM"')
assert alarme["equipamento"] == "RADIO-02"
assert alarme["momento"] == "2026-03-02 01:00:00"
print("Exercício 07: todos os testes passaram!")

### Exercício 08

Escreva `ip_valido(texto)`: `True` se o texto for um IPv4 no formato `a.b.c.d`,
com cada parte de 0 a 255; `False` caso contrário. Espaços nas pontas são aceitos.

In [ ]:
def ip_valido(texto):
    """Diz se o texto é um IPv4 a.b.c.d, cada parte de 0 a 255."""
    # TODO: strip e split("."); quatro partes; cada uma isdigit() e <= 255.
    #       Devolva False na primeira condição que falhar.
    pass

In [ ]:
# Célula de teste — Exercício 08
assert ip_valido("10.0.3.47") is True
assert ip_valido(" 192.168.1.1 ") is True
assert ip_valido("0.0.0.0") is True
assert ip_valido("255.255.255.255") is True
assert ip_valido("256.1.1.1") is False
assert ip_valido("10.0.3") is False
assert ip_valido("10.0.3.47.5") is False
assert ip_valido("10.0..47") is False
assert ip_valido("10.0.3.-1") is False
assert ip_valido("dez.0.3.47") is False
assert ip_valido("") is False
print("Exercício 08: todos os testes passaram!")

### Exercício 09

Endereços MAC chegam escritos de jeitos diferentes. Escreva `normaliza_mac(texto)`,
que aceita pares separados por `:` ou `-`, em maiúsculas ou minúsculas, e devolve o
MAC em minúsculas com dois-pontos — ou `None` se não for um MAC válido (seis pares
de caracteres hexadecimais).

```python
normaliza_mac("AC-DE-48-00-11-22")   # -> "ac:de:48:00:11:22"
normaliza_mac("ac:de:48:00:11")      # -> None
```

In [ ]:
HEXADECIMAL = "0123456789abcdef"


def normaliza_mac(texto):
    """MAC em minúsculas com ":", ou None se o texto não for um MAC."""
    # TODO: strip, lower, troque "-" por ":" e quebre em ":".
    #       Seis partes, cada uma com 2 caracteres, todos em HEXADECIMAL.
    #       No fim, junte com ":".join(partes).
    pass

In [ ]:
# Célula de teste — Exercício 09
assert normaliza_mac("AC-DE-48-00-11-22") == "ac:de:48:00:11:22"
assert normaliza_mac("ac:de:48:00:11:22") == "ac:de:48:00:11:22"
assert normaliza_mac(" Ac:De:48:00:11:22 ") == "ac:de:48:00:11:22"
assert normaliza_mac("ac:de:48:00:11") is None
assert normaliza_mac("ac:de:48:00:11:2g") is None
assert normaliza_mac("ac:de:48:00:11:222") is None
assert normaliza_mac("acde.4800.1122") is None
assert normaliza_mac("") is None
print("Exercício 09: todos os testes passaram!")

### Exercício 10

Escreva `resumo_do_log(caminho)`, que lê um log com os três formatos misturados e
devolve **dois valores**: um dicionário **severidade → quantidade** e a **lista das
linhas descartadas** (não vazias, que nenhum leitor entendeu), na ordem do arquivo.

Os três leitores e o `detecta_formato` já estão no esqueleto. Uma linha é
descartada quando o formato é `None` **ou** quando o leitor dá `IndexError`,
`KeyError` ou `ValueError` — use `try/except` **dentro** do laço. Linhas vazias são
só puladas, não contam como descartadas.

A célula de preparo cria `alarmes_misturados.log`.

In [ ]:
MESES = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
         "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
SEVERIDADE_POR_NUMERO = {"0": "EMERGENCY", "1": "ALERT", "2": "CRITICAL", "3": "ERROR",
                         "4": "WARNING", "5": "NOTICE", "6": "INFO", "7": "DEBUG"}
SEVERIDADE_C = {"crit": "CRITICAL", "warn": "WARNING", "info": "INFO"}


def le_formato_a(linha):
    """Formato A. JÁ ESCRITA."""
    campos = linha.split(maxsplit=4)
    return {"momento": campos[0] + " " + campos[1], "severidade": campos[2],
            "equipamento": campos[3], "mensagem": campos[4]}


def le_formato_b(linha, ano=2026):
    """Formato B. JÁ ESCRITA."""
    campos = linha.split(maxsplit=4)
    mes = MESES[campos[0]]
    dia = int(campos[1])
    codigo, _, mensagem = campos[4].partition(": ")
    partes = codigo.strip("%").split("-")
    return {"momento": f"{ano}-{mes:02d}-{dia:02d} {campos[2]}",
            "severidade": SEVERIDADE_POR_NUMERO[partes[1]],
            "equipamento": campos[3], "mensagem": mensagem}


def le_formato_c(linha):
    """Formato C. JÁ ESCRITA."""
    antes, _, mensagem = linha.partition(" msg=")
    campos = {}
    for par in antes.split():
        chave, _, valor = par.partition("=")
        campos[chave] = valor
    return {"momento": campos["ts"].replace("T", " "),
            "severidade": SEVERIDADE_C[campos["sev"]],
            "equipamento": campos["host"], "mensagem": mensagem.strip('"')}


def detecta_formato(linha):
    """"A", "B", "C" ou None. JÁ ESCRITA."""
    if linha.startswith("ts="):
        return "C"
    if linha[:3] in MESES and "%" in linha:
        return "B"
    if linha[:4].isdigit() and linha[4:5] == "-":
        return "A"
    return None


def resumo_do_log(caminho):
    """(contagem por severidade, linhas descartadas) de um log misturado."""
    # TODO: abra o arquivo; para cada linha: strip, pule as vazias,
    #       detecte o formato e chame o leitor certo dentro de um try.
    #       Formato None ou erro no leitor -> a linha vai para as descartadas.
    #       Senão, conte a severidade com o .get(sev, 0) + 1 da Unidade 2.
    pass

In [ ]:
# Célula de teste — Exercício 10
contagem, descartadas = resumo_do_log("alarmes_misturados.log")
assert contagem == {"CRITICAL": 3, "ERROR": 1, "WARNING": 2}, contagem
assert len(descartadas) == 4, descartadas
assert descartadas[0] == "-- MARK --"
assert descartadas[-1] == "2026-03-02 15:0"
print("Exercício 10: todos os testes passaram!")